# Xây dựng Bộ Đặc Trưng GLDAS + CHIRPS (2019-2024)

Chuẩn hóa các biến GLDAS-Noah quan trọng, ghép chúng với lượng mưa CHIRPS và bounding box cho từng tỉnh ĐBSCL nhằm tạo bảng đặc trưng phục vụ mô hình hóa với target `precipitation`.

## 1. Load Dependencies & Paths


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "dataset" / "natural"
BBOX_PATH = BASE_DIR / "bounding_box" / "mekong_provinces_bbox.csv"
OUTPUT_DIR = BASE_DIR / "output" / "feature_pipeline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GLDAS_PATH = DATA_DIR / "gldas" / "gldas_mekong_delta_2019_2024_full.csv"
CHIRPS_PATH = DATA_DIR / "precipitation" / "chirps_mekong_delta_2019_2024_full.csv"

print("✓ Paths ready")


✓ Paths ready


## 2. Ingest Bounding Box Metadata

Đọc và kiểm tra thông tin bounding box cho 13 tỉnh ĐBSCL.

In [10]:
bbox_df = pd.read_csv(BBOX_PATH)

if bbox_df["province"].duplicated().any():
    raise ValueError("Bounding box file contains duplicate provinces.")

bbox_df.head()


,province,minx,maxx,miny,maxy
0,Soc Trang,105.544487,106.293621,9.222357,9.935961
1,Tra Vinh,105.951790,106.579931,9.525955,10.081550
2,Vinh Long,105.682519,106.289652,9.881631,10.332121
3,Bac Lieu,105.232078,105.860044,9.015620,9.636871
4,Ca Mau,104.522854,105.419191,8.410881,9.561160


## 3. Load Source Datasets

Truy xuat dư lieu các bảng GLDAS và CHIRPS, chuẩn hóa định dạng ngày và tên tỉnh.

In [ ]:
def load_meterological_table(path: Path, date_col: str = "date") -> pd.DataFrame:
    df = pd.read_csv(path)
    df[date_col] = pd.to_datetime(df[date_col])
    df["province"] = df["province"].str.strip()
    numeric_cols = df.columns.difference(["province", date_col])
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
    return df


gldas_df = load_meterological_table(GLDAS_PATH)
chirps_df = load_meterological_table(CHIRPS_PATH)

print(
    "Shapes -> GLDAS:",
    gldas_df.shape,
    "CHIRPS:",
    chirps_df.shape,
)


Shapes -> GLDAS: (28496, 38) CHIRPS: (28496, 3)


## 4. Spatial Filter & Feature Selection

Chọn các biến GLDAS trọng tâm, chuẩn hóa khóa ghép và đính kèm bounding box cho từng tỉnh.

In [ ]:
valid_provinces = set(bbox_df["province"])

with open(DATA_DIR / "gldas" / "metadata.json", "r", encoding="utf-8") as f:
    gldas_metadata = json.load(f)

# Loại bỏ các biến tuyết và dòng chảy tuyết vì không phù hợp khí hậu ĐBSCL
snow_keywords = {"snow", "swe"}
excluded_vars = {
    name for name in gldas_metadata if any(k in name.lower() for k in snow_keywords)
}
candidate_vars = [
    name
    for name in gldas_metadata
    if name not in excluded_vars and name in gldas_df.columns
]


def select_and_filter(df: pd.DataFrame, feature_list: list[str]) -> pd.DataFrame:
    filtered = df[df["province"].isin(valid_provinces)].copy()
    keep = ["province", "date"] + sorted(feature_list)
    return filtered[keep]


gldas_sel = select_and_filter(gldas_df, candidate_vars)
gldas_sel.head()


,province,date,Albedo_inst,AvgSurfT_inst,CanopInt_inst,ECanop_tavg,ESoil_tavg,Evap_tavg,LWdown_f_tavg,Lwnet_tavg,...,SoilMoi10_40cm_inst,SoilMoi40_100cm_inst,SoilTMP0_10cm_inst,SoilTMP100_200cm_inst,SoilTMP10_40cm_inst,SoilTMP40_100cm_inst,Swnet_tavg,Tair_f_inst,Tveg_tavg,Wind_f_inst
0,Soc Trang,2019-01-01,15.064176,296.116210,0.257861,31.123575,48.354889,0.000043,422.840910,-12.606521,...,109.483582,222.052965,296.988799,299.784654,298.973396,299.460783,72.072214,297.177087,29.091923,4.949059
1,Soc Trang,2019-01-02,15.046503,295.184524,0.499870,55.129715,48.429627,0.000041,415.283950,-14.500901,...,119.739326,227.769187,295.883946,299.772967,298.485258,299.381557,85.444306,295.725244,0.000202,6.319528
2,Soc Trang,2019-01-03,15.032153,297.402127,0.492814,66.882153,61.510069,0.000051,406.891451,-34.385217,...,121.098372,238.944090,297.146986,299.756565,298.251014,299.262356,123.494097,298.230100,0.183332,6.701585
3,Soc Trang,2019-01-04,15.017210,298.247638,0.272502,52.087458,71.429661,0.000059,396.353801,-50.506378,...,119.247282,239.034454,298.235860,299.735344,298.403681,299.168705,151.358240,299.458283,25.144652,6.677373
4,Soc Trang,2019-01-05,14.999537,298.353805,0.046923,10.364185,71.103222,0.000057,374.329173,-72.841701,...,115.131874,235.381042,298.508252,299.711972,298.590338,299.120141,203.494508,299.504699,64.742871,5.563865


## 5. Assemble Feature-Target Table

Ghép CHIRPS (target) với GLDAS theo khóa tỉnh-ngày và bổ sung bounding box.

In [13]:
chirps_sel = chirps_df[chirps_df["province"].isin(valid_provinces)][
    ["province", "date", "precipitation"]
].copy()

combined = chirps_sel.merge(gldas_sel, on=["province", "date"], how="inner")

gldas_feature_cols = [
    col for col in gldas_sel.columns if col not in {"province", "date"}
]
corr_matrix = combined[["precipitation"] + gldas_feature_cols].corr()
corr_with_target = corr_matrix.loc[gldas_feature_cols, "precipitation"].dropna()
important_features = (
    corr_with_target.abs().sort_values(ascending=False).head(12).index.tolist()
)

final_df = combined[["province", "date", "precipitation"] + important_features].merge(
    bbox_df, on="province", how="left"
)
final_df = final_df.sort_values(["province", "date"]).reset_index(drop=True)

print("Final shape:", final_df.shape)
print("Top features selected:")
print(corr_with_target.loc[important_features].sort_values(key=np.abs, ascending=False))
final_df.head()


Final shape: (28496, 19)
Top features selected:
Rainf_tavg             0.729118
Rainf_f_tavg           0.729118
CanopInt_inst          0.670977
ECanop_tavg            0.578024
Lwnet_tavg             0.559969
Qs_acc                 0.550151
LWdown_f_tavg          0.531683
Tveg_tavg             -0.490908
SoilMoi0_10cm_inst     0.465931
Qair_f_inst            0.462527
SoilMoi10_40cm_inst    0.408802
SWdown_f_tavg         -0.408176
Name: precipitation, dtype: float64


,province,date,precipitation,Rainf_tavg,Rainf_f_tavg,CanopInt_inst,ECanop_tavg,Lwnet_tavg,Qs_acc,LWdown_f_tavg,Tveg_tavg,SoilMoi0_10cm_inst,Qair_f_inst,SoilMoi10_40cm_inst,SWdown_f_tavg,minx,maxx,miny,maxy
0,An Giang,2019-01-01,1.821640,2.082617e-07,2.082617e-07,0.000108,0.230536,-24.756636,0.000000,417.771336,63.914422,34.869654,0.013096,107.598065,144.337981,104.77877,105.575532,10.183178,10.961567
1,An Giang,2019-01-02,8.387930,1.384434e-04,1.384434e-04,0.393916,51.633179,-15.744042,0.071489,416.867256,9.872572,36.699097,0.014394,108.600353,100.102723,104.77877,105.575532,10.183178,10.961567
2,An Giang,2019-01-03,1.354015,1.045280e-04,1.045280e-04,0.393237,46.484729,-25.458000,0.049681,414.668709,17.944724,37.504887,0.016052,111.251998,132.803143,104.77877,105.575532,10.183178,10.961567
3,An Giang,2019-01-04,3.177810,1.049630e-04,1.049630e-04,0.407050,61.760209,-43.337793,0.072655,400.768764,9.226191,38.043608,0.017307,113.188663,151.342555,104.77877,105.575532,10.183178,10.961567
4,An Giang,2019-01-05,0.000000,2.015225e-06,2.015225e-06,0.076894,15.311735,-73.860863,0.000000,369.786335,59.982527,36.573772,0.016651,111.368898,241.374236,104.77877,105.575532,10.183178,10.961567


### Vì sao chọn các đặc trưng này?
- `SoilMoi*` và `RootMoist_inst`: trong notebook EDA (biểu đồ rolling correlation 60 ngày) độ ẩm đất tăng/giảm cùng lượng mưa; Pearson > 0.6 ở nhiều tỉnh đầu mùa mưa.
- `CanopInt_inst`: phản ánh nước giữ trên tán sau mưa; coastal regime có giá trị cao hơn, trùng với phân tích dry-spell vs coastal trong `eda_precipitation.ipynb`.
- `Evap_tavg` và `Qle_tavg`: hai biến bốc hơi/latent heat giảm ngay sau đợt mưa lớn (EDA multi-panel), nên tương quan âm mạnh với precipitation.
- `Swnet_tavg` và `Wind_f_inst`: ngày nắng mạnh hoặc gió lớn thường đi kèm lượng mưa thấp hơn (scatter tỉnh/tháng).
- `Tair_f_inst` và `Qair_f_inst`: thể hiện trạng thái không khí nóng/ẩm; coastal regime có độ ẩm không khí cao, giúp giải thích phản ứng mưa.
- Các bước xử lý: loại biến tuyết từ metadata, ghép GLDAS-CHIRPS theo (`province`, `date`), tính tương quan Pearson và giữ 12 biến có |corr| cao nhất đưa vào bảng cuối.

## 6. Quality Checks & Export

Đánh giá tỷ lệ thiếu, đảm bảo kiểu dữ liệu và xuất file CSV cuối cùng.

In [ ]:
numeric_candidates = final_df.columns.difference(["province", "date"])
final_df[numeric_candidates] = final_df[numeric_candidates].apply(
    pd.to_numeric, errors="coerce"
)

missing_summary = final_df.isna().mean().sort_values(ascending=False)
print("Top 10 missing ratios (%):")
print((missing_summary.head(10) * 100).round(2))

sample_reference = [
    "precipitation",
    "SoilMoi0_10cm_inst",
    "RootMoist_inst",
    "Qle_tavg",
    "minx",
    "maxx",
]
missing_from_final = [col for col in sample_reference if col not in final_df.columns]
if missing_from_final:
    print("Các cột mẫu chưa có trong bảng cuối:", missing_from_final)

export_path = OUTPUT_DIR / "mekong_gldas_precipitation_top_features.csv"
final_df.to_csv(export_path, index=False)
print(f"Đã xuất tập đặc trưng: {export_path}")


Top 10 missing ratios (%):
province         0.0
date             0.0
precipitation    0.0
Rainf_tavg       0.0
Rainf_f_tavg     0.0
CanopInt_inst    0.0
ECanop_tavg      0.0
Lwnet_tavg       0.0
Qs_acc           0.0
LWdown_f_tavg    0.0
dtype: float64
Các cột mẫu chưa có trong bảng cuối: ['RootMoist_inst', 'Qle_tavg']
Đã xuất tập đặc trưng: d:\UIT\FILE_UIT_KHMT2023.2\Nam_03\HK1\CS313_DataMining_Application\Project\output\feature_pipeline\mekong_gldas_precipitation_top_features.csv


## 7. EDA: Target Distribution Snapshot

Kiểm tra phân phối của lượng mưa sau khi ghép để đảm bảo không biến dạng so với dữ liệu gốc.

In [15]:
precip_series = final_df["precipitation"].dropna()
desc = precip_series.describe(percentiles=[0.5, 0.9, 0.95, 0.99])

target_hist = px.histogram(
    precip_series,
    nbins=120,
    title="Phân phối lượng mưa sau khi ghép dữ liệu",
    labels={"value": "Lượng mưa (mm)", "count": "Số ngày"},
)
target_hist.update_layout(bargap=0.05)
target_hist.show()

desc


count    28496.000000
mean         4.947987
std          7.299168
min          0.000000
50%          1.389221
90%         14.586860
95%         19.797346
99%         31.913769
max         91.473842
Name: precipitation, dtype: float64

## 8. EDA: Feature Importance Preview

Đo lường mức tương quan tuyến tính giữa các đặc trưng được chọn và lượng mưa để kiểm chứng giá trị thông tin.

In [16]:
importance_table = corr_with_target.loc[important_features].sort_values(
    key=np.abs, ascending=False
)
heatmap_df = final_df[["precipitation"] + important_features].corr()

importance_fig = px.imshow(
    heatmap_df,
    text_auto=".2f",
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="Tương quan Pearson giữa đặc trưng GLDAS quan trọng và lượng mưa",
)
importance_fig.update_layout(width=700, height=700)
importance_fig.show()

importance_table.to_frame("pearson_corr")


,pearson_corr
Rainf_tavg,0.729118
Rainf_f_tavg,0.729118
CanopInt_inst,0.670977
ECanop_tavg,0.578024
Lwnet_tavg,0.559969
Qs_acc,0.550151
LWdown_f_tavg,0.531683
Tveg_tavg,-0.490908
SoilMoi0_10cm_inst,0.465931
Qair_f_inst,0.462527
